# TP - Excepciones
---

## Lanzar y tratar excepción

Cree una clase con un método main() que genere un objeto de la clase Exception dentro de un bloque try. Proporcione al constructor de Exception un argumento String. Capture la excepción dentro de una cláusula catch e imprima el argumento String. Añada una cláusula finally e imprima un mensaje para demostrar que pasó por allí

In [2]:
class Tratarexcepcion():
    
    def main(self):
        try:
            raise Exception("Ocurrio un error propio")
        except Exception as e:
            print(e)
        finally:
            print("Pase por el finally")



t = Tratarexcepcion()
t.main()




Ocurrio un error propio
Pase por el finally


## Objeto nulo

Defina una referencia a un objeto e inicializela a null. Trate de invocar un método a través de esta referencia. Ahora rodee el código con una cláusula try-catch para probar la nueva excepción

In [4]:
class Persona:
    def saludar(self):
        print("Hola")

def main():
    persona = None
    try:
        persona.saludar()
    except AttributeError as ae:
        print("No se puede invocar al metodo", ae)


main()


No se puede invocar al metodo 'NoneType' object has no attribute 'saludar'


## Índice de matriz fuera de límites

Escriba código para generar y capturar una excepción ArrayIndexOutOfBoundsException

## Excepción propia

Cree su propia clase de excepción utilizando la palabra clave extends. Escriba un constructor para dicha clase que tome un argumento String y lo almacene dentro del objeto como una referencia de tipo String. Escriba un método que muestre la cadena de caracteres almacenada. Cree una cláusula try-catch para probar la nueva excepción

In [15]:
class MiExcepcion(Exception):
    def __init__(self, mensaje):
        super().__init__(mensaje)
        self.__mensaje = mensaje

    def mostrar_mensaje(self):
        print(self.__mensaje, "!!!!!!")


def main():
    try:
        raise MiExcepcion("Este es un error personalizado")
    except MiExcepcion as me:
        me.mostrar_mensaje()

main()
        

Este es un error personalizado !!!!!!


## Reanudación

Defina un comportamiento de tipo reanudación utilizando un bucle while que se repita hasta que se deje de generar una excepción

## Cuentas de un Banco

Un banco contiene las Cuentas de sus clientes. Las cuentas de ahorro no pueden tener números rojos. Las cuentas corrientes pueden tener una cuenta de ahorro asociada, de forma que si se intenta retirar más dinero del saldo actual, se debe retirar el dinero que falte de la cuenta de ahorro asociada
1. Define Cuenta de forma que no pueda instanciarse. De toda Cuenta se debe poder ingresar y retirar dinero, preguntar por el saldo, por el DNI del titular y debe tener un método toString de devuelva al menos el saldo y el DNI del titular
1. Implementa las clases CuentaCorriente y CuentaDeAhorro
1. Crea una especialización CuentaDeAhorroEsp de CuentaDeAhorro en la que se añade un entero penalización, de forma que se penaliza la retirada con una penalización % del dinero retirado. Sobrescribe sólo los métodos necesarios (incluyendo constructor y toString)
1. Si el saldo de CuentaDeAhorro fuese a quedar negativo, antes de debe lanzar una excepción SaldoNegativo (que hereda de Exception)
1. Implementa la clase Banco que contiene una colección de Cuentas, incluyendo el constructor que consideres más apropiado
1. Incluye el método totalSaldoMaxPenalización en la clase Banco que devuelva la suma de los saldos de todas las cuentas corrientes y la máxima penalización entre las CuentaDeAhorroEsp
1. Crea una clase con método main en la que instanciamos un Banco de 100 Cuentas, nos creamos una CuentaCorriente con 5000 pesos y una cuenta de ahorro especial con una penalización del 5%. Finalmente debe mostrar información de todas las Cuentas del Banco. En un bloque try & catch retirar dinero de una Cuenta de Ahorro

In [1]:
##1. Excepción propia y clase abstracta Cuenta

from abc import ABC, abstractmethod


class SaldoNegativo(Exception):
    pass


class Cuenta(ABC):

    def __init__(self, dni, saldo=0):
        if saldo < 0:
            raise ValueError("El saldo inicial no puede ser negativo")

        self.__dni = dni
        self.__saldo = saldo

    @property
    def dni(self):
        return self.__dni

    @property
    def saldo(self):
        return self.__saldo

    def ingresar(self, cantidad):
        if cantidad <= 0:
            raise ValueError("El ingreso debe ser positivo")

        self.__saldo += cantidad

    def _actualizar_saldo(self, nuevo_saldo):
        self.__saldo = nuevo_saldo

    @abstractmethod
    def retirar(self, cantidad):
        pass

    def __str__(self):
        return f"DNI: {self.dni} | Saldo: ${self.saldo:.2f}"
        
        
        
        
##2. Cuenta de ahorro

class CuentaDeAhorro(Cuenta):

    def retirar(self, cantidad):
        if cantidad <= 0:
            raise ValueError("El retiro debe ser positivo")

        if cantidad > self.saldo:
            raise SaldoNegativo(
                "No hay saldo suficiente en la cuenta de ahorro"
            )

        self._actualizar_saldo(self.saldo - cantidad)
        
        
        
## 3. Cuenta de ahorro especial

class CuentaDeAhorroEsp(CuentaDeAhorro):

    def __init__(self, dni, saldo, penalizacion):
        super().__init__(dni, saldo)

        if not 0 <= penalizacion <= 100:
            raise ValueError(
                "La penalización debe estar entre 0 y 100"
            )

        self.__penalizacion = penalizacion

    @property
    def penalizacion(self):
        return self.__penalizacion

    def retirar(self, cantidad):
        if cantidad <= 0:
            raise ValueError("El retiro debe ser positivo")

        total = cantidad * (1 + self.penalizacion / 100)

        if total > self.saldo:
            raise SaldoNegativo(
                "El saldo no alcanza para cubrir el retiro y la penalización"
            )

        self._actualizar_saldo(self.saldo - total)

    def __str__(self):
        return (
            f"{super().__str__()} | "
            f"Penalización: {self.penalizacion}%"
        )
        
        
        
        
## 4. Cuenta corriente
class CuentaCorriente(Cuenta):

    def __init__(self, dni, saldo=0, cuenta_ahorro=None):
        super().__init__(dni, saldo)
        self.__cuenta_ahorro = cuenta_ahorro

    def asociar_ahorro(self, cuenta_ahorro):
        self.__cuenta_ahorro = cuenta_ahorro

    def retirar(self, cantidad):
        if cantidad <= 0:
            raise ValueError("El retiro debe ser positivo")

        if cantidad <= self.saldo:
            self._actualizar_saldo(self.saldo - cantidad)
            return

        faltante = cantidad - self.saldo

        if self.__cuenta_ahorro is None:
            raise SaldoNegativo(
                "No hay saldo suficiente ni una cuenta de ahorro asociada"
            )

        # Primero comprobamos que el ahorro pueda cubrir el faltante.
        # Si falla, el saldo corriente todavía no se modificó.
        self.__cuenta_ahorro.retirar(faltante)

        self._actualizar_saldo(0)

    def __str__(self):
        return (
            f"Cuenta corriente | {super().__str__()} | "
            f"Ahorro asociado: "
            f"{'Sí' if self.__cuenta_ahorro else 'No'}"
        )
        
        
## 5. Clase Banco

class Banco:

    def __init__(self, capacidad=100):
        if capacidad <= 0:
            raise ValueError("La capacidad debe ser positiva")

        self.__capacidad = capacidad
        self.__cuentas = []

    def agregar_cuenta(self, cuenta):
        if len(self.__cuentas) >= self.__capacidad:
            raise ValueError("El banco alcanzó su capacidad máxima")

        if not isinstance(cuenta, Cuenta):
            raise TypeError("Solo se pueden agregar cuentas bancarias")

        self.__cuentas.append(cuenta)

    def totalSaldoMaxPenalizacion(self):
        total_saldo = sum(
            cuenta.saldo
            for cuenta in self.__cuentas
            if isinstance(cuenta, CuentaCorriente)
        )

        penalizaciones = [
            cuenta.penalizacion
            for cuenta in self.__cuentas
            if isinstance(cuenta, CuentaDeAhorroEsp)
        ]

        max_penalizacion = max(penalizaciones, default=0)

        return total_saldo, max_penalizacion

    def mostrar_cuentas(self):
        for cuenta in self.__cuentas:
            print(cuenta)

    def __str__(self):
        return (
            f"Banco: {len(self.__cuentas)} cuentas "
            f"registradas de una capacidad de {self.__capacidad}"
        )
        
 
 
 
## 6 Programa Principal

banco = Banco(100)

ahorro = CuentaDeAhorroEsp(
    dni="30111222",
    saldo=2000,
    penalizacion=5
)

corriente = CuentaCorriente(
    dni="30111222",
    saldo=5000,
    cuenta_ahorro=ahorro
)

banco.agregar_cuenta(corriente)
banco.agregar_cuenta(ahorro)

# Probamos un retiro de la cuenta de ahorro.
try:
    ahorro.retirar(2500)
    print("Retiro realizado correctamente")

except SaldoNegativo as e:
    print("No se pudo realizar el retiro:", e)

print("\nInformación del banco:")
print(banco)
banco.mostrar_cuentas()

total_saldo, max_penalizacion = (
    banco.totalSaldoMaxPenalizacion()
)

print("\nResumen:")
print(f"Total de saldos en cuentas corrientes: ${total_saldo:.2f}")
print(f"Máxima penalización: {max_penalizacion}%")






No se pudo realizar el retiro: El saldo no alcanza para cubrir el retiro y la penalización

Información del banco:
Banco: 2 cuentas registradas de una capacidad de 100
Cuenta corriente | DNI: 30111222 | Saldo: $5000.00 | Ahorro asociado: Sí
DNI: 30111222 | Saldo: $2000.00 | Penalización: 5%

Resumen:
Total de saldos en cuentas corrientes: $5000.00
Máxima penalización: 5%
